<a href="https://colab.research.google.com/github/TrinaBan0807/python-starter-kit/blob/main/Clip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install open_clip_torch
import torch
import open_clip
import copy
from PIL import Image
import torch.nn.functional as F
import numpy as np

# --- 1. Setup and CLIP Loading ---
print("Loading CLIP model...")
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()
print("CLIP model loaded.")

# --- 2. Refined Label Taxonomy (Class Design) ---
# Expanding labels to provide richer semantic context
print("\n--- Demonstrating Refined Label Taxonomy ---")
dummy_image_features = torch.randn(3, model.visual.output_dim)
dummy_image_features = dummy_image_features / dummy_image_features.norm(dim=-1, keepdim=True)

# Original labels vs refined context-rich labels
original_labels = ["cat", "dog", "automobile", "truck"]
refined_labels = ["Siamese cat", "Golden Retriever", "sports car", "dump truck", "child", "person"]

print(f"Original Labels: {original_labels}")
print(f"Refined Labels: {refined_labels}")

# --- 3. Weight Interpolation Tweak ---
# Addressing the zero-shot to few-shot performance drop
print("\n--- Applying Weight Interpolation ---")

# Access the projection parameter directly (Parameter tensor)
original_text_projection = model.text_projection.clone().detach()

# Simulate 'adapted' weights from few-shot learning
adapted_text_projection = original_text_projection + 0.01 * torch.randn_like(original_text_projection)

def interpolate_weights(theta_0, theta_few, alpha):
    """
    Blends zero-shot and few-shot weights.
    Formula: (1 - alpha) * theta_0 + alpha * theta_few
    """
    return (1 - alpha) * theta_0 + alpha * theta_few

# Alpha=0.3 preserves the zero-shot prior while accepting few-shot updates
alpha = 0.3
tweaked_model = copy.deepcopy(model)

with torch.no_grad():
    interpolated_weight = interpolate_weights(
        original_text_projection,
        adapted_text_projection,
        alpha
    )
    # Correctly copy the interpolated tensor back to the model parameter
    tweaked_model.text_projection.copy_(interpolated_weight)

print(f"Weights successfully interpolated with alpha = {alpha}.")

# --- 4. Final Verification ---
refined_text_features = tweaked_model.encode_text(tokenizer(refined_labels))
refined_text_features = refined_text_features / refined_text_features.norm(dim=-1, keepdim=True)
logits = (dummy_image_features @ refined_text_features.T).softmax(dim=-1)

print("\nMethodological Summary:")
print("1. Class Design: Taxonomy expanded to resolve specialized tasks and mitigate bias.")
print("2. Weight Interpolation: Zero-shot weights used as a prerequisite to stabilize few-shot learning.")

Loading CLIP model...
CLIP model loaded.

--- Demonstrating Refined Label Taxonomy ---
Original Labels: ['cat', 'dog', 'automobile', 'truck']
Refined Labels: ['Siamese cat', 'Golden Retriever', 'sports car', 'dump truck', 'child', 'person']

--- Applying Weight Interpolation ---
Weights successfully interpolated with alpha = 0.3.

Methodological Summary:
1. Class Design: Taxonomy expanded to resolve specialized tasks and mitigate bias.
2. Weight Interpolation: Zero-shot weights used as a prerequisite to stabilize few-shot learning.
